In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../../data/raw/RecipeNLG_dataset.csv")
# df = pd.read_csv("")

In [3]:
df.head(1)

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."


In [4]:
print(df.loc[0, "ingredients"])
print(type(df.loc[0, "ingredients"]))

print(df.loc[0, "NER"])

["1 c. firmly packed brown sugar", "1/2 c. evaporated milk", "1/2 tsp. vanilla", "1/2 c. broken nuts (pecans)", "2 Tbsp. butter or margarine", "3 1/2 c. bite size shredded rice biscuits"]
<class 'str'>
["brown sugar", "milk", "vanilla", "nuts", "butter", "bite size shredded rice biscuits"]


In [5]:
df = df.sample(1000000, random_state=42)

In [6]:
import ast

def safe_parse(x):
    try:
        return ast.literal_eval(x)
    except:
        return []

df["ingredients_list"] = df["ingredients"].apply(safe_parse)
df["steps_list"] = df["directions"].apply(safe_parse)
df["ner_list"] = df["NER"].apply(safe_parse)

In [7]:
df[["ingredients_list", "steps_list", "ner_list"]].head(5)

,ingredients_list,steps_list,ner_list
2015528,"[1 1/2 pound flank steak, 1/2 c. finely minced...","[Remove tenderloin from steak., Score meat., C...","[flank steak, green onions, red wine, soy sauc..."
1608734,"[1 tablespoon rosemary, 1 teaspoon thyme, 3 ba...",[combine all ingredients in slow cooker (6 qua...,"[rosemary, thyme, bay leaves, paprika, pepper,..."
778500,"[3 to 4 carrots, 1 1/2 Tbsp. butter, 1/3 c. br...",[Cook 3 to 4 carrots; cut crosswise in 1-inch ...,"[carrots, butter, brown sugar, lemon rind]"
1334975,"[4.5 Cups Flour, 1.5 Tsp Salt, Pinch Baking Po...","[Mix all dry ingredients in a bowl., , Add cri...","[Flour, Salt, Baking Powder, Sugar, Crisco, eg..."
116562,"[2 c. crushed small thin pretzels (sticks), 3/...","[Mix and press in baking pan, approximately 13...","[thin pretzels, margarine]"


In [8]:
def clean_text(text):
    return str(text).strip().lower()


In [9]:
df["title"] = df["title"].apply(clean_text)

df["ingredients_clean"] = df["ner_list"].apply(
    lambda lst: [clean_text(i) for i in lst if i]
)

df["steps_clean"] = df["steps_list"].apply(
    lambda lst: [clean_text(i) for i in lst if i]
)

In [10]:
df_clean = df[
    (df["ingredients_clean"].map(len) > 2) &
    (df["steps_clean"].map(len) > 1)
]

In [11]:
import uuid

def build_record(row):
    return {
        "id": str(uuid.uuid4()),
        "title": row["title"],
        "ingredients": row["ingredients_clean"],
        "steps": row["steps_clean"]
    }

final_data = df_clean.apply(build_record, axis=1).tolist()

In [12]:
final_data[0]

{'id': 'b70eab54-adad-4a22-9884-a61bd5a80806',
 'title': 'marinated flank steak recipe',
 'ingredients': ['flank steak',
  'green onions',
  'red wine',
  'soy sauce',
  'salad oil',
  'sesame seeds',
  'brown sugar',
  'grnd black pepper',
  'grnd ginger',
  'clove garlic'],
 'steps': ['remove tenderloin from steak.',
  'score meat.',
  'combine remaining ingredients and pour over meat.',
  'let marinate 24 hrs.',
  'preheat grill.',
  'broil or possibly grill.',
  'slice thinly on an angle against the grain.']}

In [13]:
import json

with open("../../data/processed/recipes_cleaned.json", "w") as f:
    json.dump(final_data, f, indent=2)

print(f"Saved {len(final_data)} recipes")

Saved 906142 recipes
